# Per-prior non-centered parameterization

Hierarchical models in Bambi can use the **non-centered parameterization** for group-specific effects with random hyperpriors — rewriting `b ~ Normal(0, sigma)` as `z ~ Normal(0, 1); b = z * sigma` to improve sampling geometry.

Historically this was a single model-wide knob (`Model(..., noncentered=True)`). It is now also configurable **per `bmb.Prior`** *and* **per likelihood parameter** (via `Model(noncentered={"mu": True, "sigma": False})`), so users can mix centered and non-centered parameterizations across the group-specific terms of a single model. This is most useful in distributional models, where one likelihood parameter may benefit from non-centering while another does not.

In [1]:
import bambi as bmb
import numpy as np
import pandas as pd

## Setup — the sleepstudy dataset

A small hierarchical dataset where reaction time depends linearly on days of sleep deprivation, with an intercept and a slope that vary by subject.

In [2]:
data = bmb.load_data("sleepstudy")
data.head()

,Reaction,Days,Subject
0,249.5600,0,308
1,258.7047,1,308
2,250.8006,2,308
3,321.4398,3,308
4,356.8519,4,308


## The model-level knob (legacy behavior)

`Model(..., noncentered=True)` (the default) turns on the offset trick for every group-specific term that has a random `sigma` hyperprior. Inspecting the PyMC graph after `model.build()` shows the `_offset` variables this introduces.

In [3]:
def offset_vars(model):
    return sorted(v for v in model.backend.model.named_vars if v.endswith("_offset"))

m_nc = bmb.Model("Reaction ~ Days + (Days | Subject)", data, noncentered=True)
m_nc.build()
offset_vars(m_nc)

['1|Subject_offset', 'Days|Subject_offset']

Disabling at the model level removes them entirely:

In [4]:
m_c = bmb.Model("Reaction ~ Days + (Days | Subject)", data, noncentered=False)
m_c.build()
offset_vars(m_c)

[]

## New: per-prior override

`bmb.Prior` now accepts a `noncentered=` keyword. `None` (the default) inherits the model-level value; `True` / `False` overrides it for that specific prior.

Below we keep the intercept-by-subject term non-centered and force the slope-by-subject term to use the centered parameterization. We deliberately set `Model(noncentered=False)` so the inheritance path is also exercised: the intercept prior's explicit `noncentered=True` wins, and the slope prior inherits the model default.

In [5]:
hyper = lambda nc: bmb.Prior(
    "Normal",
    mu=0,
    sigma=bmb.Prior("HalfNormal", sigma=1),
    noncentered=nc,
)

priors = {
    "1|Subject": hyper(True),    # force noncentered
    "Days|Subject": hyper(False),  # force centered
}

m_mixed = bmb.Model(
    "Reaction ~ Days + (Days | Subject)",
    data,
    priors=priors,
    noncentered=False,  # the inherited model-level default does not matter here
)
m_mixed.build()
offset_vars(m_mixed)

['1|Subject_offset']

Only the intercept-by-subject term has an `_offset` companion: per-prior beats the model default in both directions.

## Distributional models — independent control across parameters

When more than one likelihood parameter has its own linear predictor, each parameter is a separate component with its own group-specific terms. The per-prior flag lets you choose the parameterization independently for each.

A common case: a Gaussian distributional model where both the mean (`mu`) and the residual scale (`sigma`) get a per-subject random effect. The mean's hierarchical effect may benefit from non-centering (many subjects, sparse data per subject), while the scale's effect is often estimated well under the centered parameterization. The two can be specified independently.

### How formulas map to components

A short note before we go further. For a Gaussian family the likelihood has parameters `mu` and `sigma`. Bambi treats each as a separate **component** of the model. With a single-formula constructor (`bmb.Model("y ~ x", data)`) the response-side formula goes onto the **parent** parameter — `mu` for a Gaussian — and any auxiliary parameter (`sigma`) becomes a `ConstantComponent`: a single scalar random variable drawn from a prior, with no design matrix and no group-specific terms. To put a regression (and potentially group-specific terms) on an auxiliary parameter you use the `bmb.Formula("y ~ ...", "sigma ~ ...")` multi-formula constructor.

The `noncentered` setting only does work for components that actually have group-specific terms with random hyperpriors. Setting `noncentered={"sigma": False}` for a model whose `sigma` is a `ConstantComponent` is harmless (and accepted by the validator, because `"sigma"` is a valid component name) but a no-op — there are no group-specific terms in that component to govern.

In [6]:
formula = bmb.Formula(
    "Reaction ~ 1 + (1 | Subject)",
    "sigma ~ 1 + (1 | Subject)",
)

# Per-prior `noncentered=` is set independently on each component's prior.
parent_prior = bmb.Prior(
    "Normal",
    mu=0,
    sigma=bmb.Prior("HalfNormal", sigma=1),
    noncentered=True,   # parent component (mu): non-centered
)
sigma_prior = bmb.Prior(
    "Normal",
    mu=0,
    sigma=bmb.Prior("HalfNormal", sigma=1),
    noncentered=False,  # auxiliary component (sigma): centered
)

priors = {
    "1|Subject": parent_prior,                  # parent (mu) component
    "sigma": {"1|Subject": sigma_prior},        # auxiliary component
}

m_dist = bmb.Model(formula, data, priors=priors)
m_dist.build()
offset_vars(m_dist)


['1|Subject_offset']

## Per-component shortcut: `Model(noncentered={...})`

When a distributional model has several response parameters and you want a different *default* parameterization per parameter, the per-`Prior` form above can become verbose if every term in the same component should share the same setting. `Model.noncentered` therefore also accepts a `dict[str, bool]` keyed by **component name** (the parameter name in the family's likelihood — `"mu"` and `"sigma"` for a Gaussian, etc.).

Missing keys fall back to `True` (the historical default). Per-`Prior` `noncentered=` still overrides the dict entry for any specific term. Unknown keys raise immediately with a list of valid component names — typos are caught at model construction, not silently.

In [7]:
m_dict = bmb.Model(
    formula,
    data,
    noncentered={"mu": True, "sigma": False},  # mu: non-centered, sigma: centered
)
m_dict.build()
offset_vars(m_dict)

['1|Subject_offset']

## Non-Normal priors with `noncentered=False`

The previous behavior was to raise `NotImplementedError` whenever a group-specific term had a non-Normal prior with a random hyperprior, even if the user only wanted the centered parameterization. With explicit `noncentered=False` on the prior, this now works — the centered branch is general.

`noncentered=True` on a non-Normal prior still raises, with an informative message naming the offending prior and pointing at the remediation.

In [8]:
st_prior = bmb.Prior(
    "StudentT",
    nu=4,
    mu=0,
    sigma=bmb.Prior("HalfNormal", sigma=1),
    noncentered=False,
)

m_st = bmb.Model(
    "Reaction ~ Days + (Days | Subject)",
    data,
    priors={"Days|Subject": st_prior},
)
m_st.build()  # used to raise NotImplementedError; now builds via the centered branch
offset_vars(m_st)

['1|Subject_offset']

## Sampling a model with each parameterization

Both parameterizations describe the same probabilistic model — they should yield the same posterior in expectation, but the sampler's behavior can differ. To make the per-component `noncentered` dict meaningful we use a **distributional formula** that gives both `mu` and `sigma` their own linear predictor with a per-subject random intercept; otherwise `sigma` would default to a `ConstantComponent` (a single scalar prior, no group-specific terms) and its entry in the dict would be a no-op.

Below we fit the same distributional model twice on the same data and seed: once with both components non-centered (the default), once with both centered via `noncentered={"mu": False, "sigma": False}`. We compare the posterior of the population coefficients and the divergence count.

In [9]:
import arviz as az

# Distributional formula: a regression for *both* mu (the response) and sigma.
# Each component then has its own group-specific intercept by Subject.
dist_formula = bmb.Formula(
    "Reaction ~ Days + (Days | Subject)",
    "sigma ~ 1 + (1 | Subject)",
)

FIT_KW = dict(tune=500, draws=500, chains=2, random_seed=1234, progressbar=False)

# Non-centered for both components.
m_nc_fit = bmb.Model(dist_formula, data, noncentered={"mu": True, "sigma": True})
idata_nc = m_nc_fit.fit(**FIT_KW)

# Centered for both components.
m_c_fit = bmb.Model(dist_formula, data, noncentered={"mu": False, "sigma": False})
idata_c = m_c_fit.fit(**FIT_KW)

# Sanity-check that each fit actually picked up the parameterization we asked for.
nc_offsets = sorted(v for v in m_nc_fit.backend.model.named_vars if v.endswith("_offset"))
c_offsets = sorted(v for v in m_c_fit.backend.model.named_vars if v.endswith("_offset"))
print("non-centered offsets:", nc_offsets)
print("centered     offsets:", c_offsets)

div_nc = int(idata_nc.sample_stats["diverging"].sum().item())
div_c = int(idata_c.sample_stats["diverging"].sum().item())
print(f"\ndivergences: non-centered={div_nc}, centered={div_c}")

print("\n--- Population coefficients (mu component) ---")
print("non-centered:")
print(az.summary(idata_nc, var_names=["Intercept", "Days"], kind="stats", round_to=2))
print("\ncentered:")
print(az.summary(idata_c, var_names=["Intercept", "Days"], kind="stats", round_to=2))

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [Intercept, Days, 1|Subject_sigma, 1|Subject_offset, Days|Subject_sigma, Days|Subject_offset, sigma_Intercept, sigma_1|Subject_sigma, sigma_1|Subject_offset]


Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


...

Multiprocess sampling (2 chains in 2 jobs)


NUTS: [Intercept, Days, 1|Subject_sigma, 1|Subject, Days|Subject_sigma, Days|Subject, sigma_Intercept, sigma_1|Subject_sigma, sigma_1|Subject]


Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


non-centered offsets: ['1|Subject_offset', 'Days|Subject_offset', 'sigma_1|Subject_offset']
centered     offsets: []

divergences: non-centered=0, centered=0

--- Population coefficients (mu component) ---
non-centered:
             mean    sd  eti89_lb  eti89_ub
Intercept  252.33  7.59    240.79    264.13
Days        10.33  1.75      7.66     13.08

centered:
             mean    sd  eti89_lb  eti89_ub
Intercept  251.36  7.15    240.12    262.59
Days        10.39  1.74      7.52     13.12


The `_offset` print confirms that the non-centered fit produced one offset RV per group-specific term in each component (`1|Subject_offset`, `Days|Subject_offset`, and the `sigma`-prefixed counterparts), while the centered fit produced none — exactly the difference between the two parameterizations.

Posterior means and credible intervals for `Intercept` and `Days` should agree closely between the two fits — the parameterization is a sampling-geometry choice, not a different model. The divergence count is parameterization-dependent and is one of the diagnostics that motivates choosing one form over the other for a given dataset.

## Summary

- `Model(..., noncentered=...)` remains the model-wide default.
- `bmb.Prior(..., noncentered=...)` overrides it per group-specific term.
- The combination lets users mix parameterizations within a single model — across grouping terms and across distributional components.
- `noncentered=False` is now general: any prior, including non-Normal hyperpriors, builds cleanly under the centered parameterization.